<a href="https://colab.research.google.com/github/bsalami-092/Data_Science_Journey_Documentation/blob/main/Geospatial_Data_Analysis_Assignment_2_with_Python_(Geocoding).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step I: Import necessary libraries
from geopy.geocoders import GoogleV3
from geopy.exc import GeocoderTimedOut
import pandas as pd
from google.colab import userdata

In [ ]:
# Step 2: Set up google maps API securely using Colab Secrets

API_KEY = userdata.get('GOOGLE_API_KEY')
geolocator = GoogleV3(api_key=API_KEY)

In [ ]:
# Step 3: Create a list of addresses
addresses = [
    "12A, Adeola Odeku Street, Victoria Island, Lagos State",
    "45, Opebi Road, Ikeja, Lagos State",
    "7, Balogun Street, Lagos Island, Lagos State",
    "23, Admiralty Way, Lekki Phase 1, Lagos State",
    "18, Ajose Adeogun Street, Victoria Island, Lagos State",
    "10, Gana Street, Maitama, Abuja FCT",
    "32, Aminu Kano Crescent, Wuse II, Abuja FCT",
    "5, Ebitu Ukiwe Street, Jabi, Abuja FCT",
    "20, Ahmadu Bello Way, Garki II, Abuja FCT",
    "14, Usuma Street, Asokoro, Abuja FCT",
    "22, Bodija Estate Road, Ibadan, Oyo State",
    "5, Queen Elizabeth Road, Mokola, Ibadan, Oyo State",
    "9, Aba Road, Port Harcourt, Rivers State",
    "17, Rumuola Road, Port Harcourt, Rivers State",
    "8, Zoo Road, Kano, Kano State",
    "21, Bompai Road, Nassarawa GRA, Kano, Kano State",
    "12, Chime Avenue, New Haven, Enugu, Enugu State",
    "33, Ogui Road, Enugu, Enugu State",
    "14, Ahmadu Bello Way, Kaduna North, Kaduna State",
    "6, Airport Road, Benin City, Edo State"
]


In [ ]:
# Step 4: Function to Geocode Addresses
def geocode_addresses(addresses):
  try:
    location = geolocator.geocode(addresses)
    if location:
      return location.latitude, location.longitude
    else:
      return None, None
  except GeocoderTimedOut:
    return None, None

In [ ]:
# Step 5: Geocode All Addresses
geo_data = [ ]

for address in addresses:
  latitude, longitude = geocode_addresses(address)
  geo_data.append({'Address': address, 'Latitude': latitude, 'Longitude': longitude})



df = pd.DataFrame(geo_data)


GeocoderQueryError: You must enable Billing on the Google Cloud Project at https://console.cloud.google.com/project/_/billing/enable Learn more at https://developers.google.com/maps/gmp-get-started

Method 2: Geocode Using Nominatim(OpenStreetMap)

Nominatim is free and does not require any API key

In [ ]:
from geopy.geocoders import Nominatim
import time

In [ ]:
geolocator = Nominatim(user_agent='geo_colab')

geo_data = [ ]

for address in addresses:
  try:
    latitude, longitude = geocode_addresses(address)
    geo_data.append({'Address': address, 'Latitude': latitude, 'Longitude': longitude})
    time.sleep(1) # Add a 1-second delay between requests
  except Exception as e:
    print(f"Error geocoding {address}: {e}")
    geo_data.append({'Address': address, 'Latitude': None, 'Longitude': None})


df = pd.DataFrame(geo_data)

In [ ]:
df.head()

,Address,Latitude,Longitude
0,"12A, Adeola Odeku Street, Victoria Island, Lag...",6.430594,3.416426
1,"45, Opebi Road, Ikeja, Lagos State",6.591096,3.359904
2,"7, Balogun Street, Lagos Island, Lagos State",6.458870,3.386269
3,"23, Admiralty Way, Lekki Phase 1, Lagos State",NaN,NaN
4,"18, Ajose Adeogun Street, Victoria Island, Lag...",6.431007,3.436418


In [ ]:
df.isna().sum()

,0
Address,0
Latitude,7
Longitude,7


* **Geocoding Addresses in CSV**
* Step 1: Load the CSV file containing addresses
* We will first create a CSV file containing the addresses

In [ ]:
# Since we already have addresses,  create a  DataFrame
data = pd.DataFrame({'Address': addresses})

# Save to CSV
csv_file = '/content/nigeria_addresses.csv'
data.to_csv(csv_file, index=False)


Step 2: Geocode Addresses Using Nominatim

In [ ]:
# Now, let use gepy to get the latitude and longitude of each addresses
geolocator = Nominatim(user_agent='nigeria_geocoder')

# Function to geocode without error handling
def geocode_address(addresses):
  try:
    location = geolocator.geocode(addresses, timeout=10)
    if location:
      return location.latitude, location.longitude
    else:
      return None, None
  except GeocoderTimedOut:
    time.sleep(2)
    return geocode_address(addresses)


# Load the csv file
data = pd.read_csv(csv_file)

# Applying geocoding
data['Latitude'], data['Longitude'] = zip(*df['Address'].apply(geocode_address))

# Save the results
output_csv = '/content/nigeria_geocoded_addresses.csv'
data.to_csv(output_csv, index=False)


data.head()

,Address,Latitude,Longitude
0,"12A, Adeola Odeku Street, Victoria Island, Lag...",6.430594,3.416426
1,"45, Opebi Road, Ikeja, Lagos State",6.591096,3.359904
2,"7, Balogun Street, Lagos Island, Lagos State",6.458870,3.386269
3,"23, Admiralty Way, Lekki Phase 1, Lagos State",NaN,NaN
4,"18, Ajose Adeogun Street, Victoria Island, Lag...",6.431007,3.436418


In [ ]:
data.isna().sum()

,0
Address,0
Latitude,7
Longitude,7


Step 3: Download the geocoded csv

In [ ]:
from google.colab import files
files.download(output_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Step 4: Install and Import Required Libraries

In [ ]:
import folium
from folium.plugins import MarkerCluster

Step 5: Load the Geocoded CSV File

* I prefer the nigeria_addresses.csv

In [ ]:
# Load  the geocoded data
csv_file = '/content/nigeria_geocoded_addresses.csv'
df = pd.read_csv(csv_file)

df.head()

,Address,Latitude,Longitude
0,"12A, Adeola Odeku Street, Victoria Island, Lag...",6.430594,3.416426
1,"45, Opebi Road, Ikeja, Lagos State",6.591096,3.359904
2,"7, Balogun Street, Lagos Island, Lagos State",6.458870,3.386269
3,"23, Admiralty Way, Lekki Phase 1, Lagos State",NaN,NaN
4,"18, Ajose Adeogun Street, Victoria Island, Lag...",6.431007,3.436418


Step 6: Create a Folium Map

Now, initialize the map centered at Lagos, Nigeria

In [ ]:
# Create a folium map centered around Lagos
m = folium.Map(location=[6.5244, 3.3792], zoom_start=6)

# Add a marker cluster to group nearby points
marker_cluster = MarkerCluster().add_to(m)

# Loop through the DataFrame and add markers
for index, row in df.iterrows():
  lat, lon = row['Latitude'], row['Longitude']
  address = row['Address']

  # Check if latitude and longitude are valid before adding marker
  if not pd.isna(lat) and not pd.isna(lon):
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(f'<b> {address}<b>', max_width=300),
        icon=folium.Icon(color='green', icon='info-sign'),
        tooltip=address
    ).add_to(marker_cluster)

else:
  print(f"Skipping marker for {address} due to missing coordinates.")


m

Skipping marker for 6, Airport Road, Benin City, Edo State due to missing coordinates.


Step 7: If you want , you can save and download the map


In [ ]:
map_filename = '/content/nigeria_pol_map.html'
m.save(map_filename)
files.download(map_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>